# paper_results010 — corrected controlled replication of the successful 005 analysis

This notebook starts from the exact `paper_results005.ipynb` source used in the successful run at commit:

`53b358b458ca4ad5fe3c5ffab85cfbd6443ab733`

It preserves the old 005 cohort as the replication reference while correcting analytical bugs identified in 005/009.

Key corrections in 010:

- NERO vectors must contain exactly 41 components; malformed vectors are excluded rather than feature-imputed.
- Derived NERO features are computed only from explicit `rates_*` columns, never by positional slicing.
- `nzero` is computed only from NERO rates and therefore cannot include the class label.
- Zero-shot NERO AUC is computed from one scalar score (`nz_entropy_rate`), not from an envelope formed by concatenating multiple ROC curves.
- Missing detector scores are excluded detector-by-detector rather than filled with zero.
- The trained classifier keeps the intended GPT-5 future holdout; the split is not overwritten later.
- Imputation is fit on training data only and applied to test data with `transform`.
- Subset results are treated as descriptive sensitivity distributions; overlapping subsets are not assigned Student-t confidence intervals.
- arXiv matching is conservative: unique YYMM matches may refresh the old row, but ambiguous YYMM groups retain the old reference row until source provenance resolves exact identity. No ambiguous alternative is labeled "genuinely new".

The 150,000-character rule is applied to newly considered current rows/replacements, but the frozen old 005 cohort is not retroactively deleted when current metadata reports a shorter text. This distinction is explicit below.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import warnings
import glob
warnings.filterwarnings('ignore')
from jupyterthemes import jtplot
jtplot.style()
from sklearn import metrics

from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn import svm
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, roc_auc_score, auc
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
import random

def pyRandColor():
    randNums = [random.random() for _ in range(0, 3)]

    RGB255 = list([ int(i * 255) for i in randNums ])
    RGB1 = list([ round(i, 2) for i in randNums ])
    return RGB1

## Inclusion rule — before any data loading

The paper cohort uses texts of at least **150,000 characters**.

For a current table that contains `text_length`, a current row below this threshold cannot replace its old reference row. The reference tables themselves were the tables used in the successful 005 run.


In [ ]:
REFERENCE_COMMIT = "53b358b458ca4ad5fe3c5ffab85cfbd6443ab733"
MIN_TEXT_LENGTH = 150_000
EXPECTED_RATE_COMPONENTS = 41

# General replication behavior for non-arXiv datasets:
# True  = same OLD cohort, refreshed CURRENT values where the same row exists.
# False = exact old values from the successful 005 snapshot.
USE_CURRENT_REPLICATE_VALUES = True

# arXiv:
# Only unambiguous YYMM matches are eligible for current-value refresh.
# Ambiguous YYMM groups stay frozen at the old reference row until provenance is known.
USE_CURRENT_ARXIV_VALUES = True

# Conservative in 010: do not add unmatched/ambiguous arXiv alternatives as "new"
# until document identity is established from source provenance.
KEEP_NEW_ARXIV_ROWS = False

print("Reference commit:", REFERENCE_COMMIT)
print("Minimum text length:", MIN_TEXT_LENGTH)
print("Expected NERO rate components:", EXPECTED_RATE_COMPONENTS)
print("Use current values for matched replicate rows:", USE_CURRENT_REPLICATE_VALUES)
print("Use current arXiv values for unambiguous matches:", USE_CURRENT_ARXIV_VALUES)
print("Keep new arXiv rows:", KEEP_NEW_ARXIV_ROWS)


## Data

In [ ]:
def getDataFrame(df, varname='human', cols=['perplexity', 'percentage', 'entropy_rate', 'rates']):
    """Load detector outputs while requiring a complete 41-component NERO vector."""
    import ast
    work = df[cols].copy()
    work = work.dropna(subset=["rates"])

    def parse_complete_rates(v):
        try:
            x = v if isinstance(v, (list, tuple, np.ndarray)) else ast.literal_eval(str(v))
            x = np.asarray(x, dtype=float)
        except Exception:
            return None
        if x.ndim != 1 or len(x) != EXPECTED_RATE_COMPONENTS:
            return None
        return x.tolist()

    parsed = work["rates"].apply(parse_complete_rates)
    malformed = parsed.isna()
    if malformed.any():
        print(f"{varname}: dropping {int(malformed.sum())} rows with malformed/incomplete NERO vectors")
    work = work.loc[~malformed].copy()
    work["rates"] = parsed.loc[~malformed]

    rates_df = pd.DataFrame(
        work["rates"].tolist(),
        index=work.index,
        columns=[f"rates_{i}" for i in range(EXPECTED_RATE_COMPONENTS)],
    )
    return pd.concat([work.drop(columns=["rates"]), rates_df], axis=1).assign(label=varname)


from pathlib import Path
from io import StringIO
import subprocess
import re
import ast

REWRITE_RE = re.compile(r"_(?:0\.9|1\.05|1\.1|1\.5)(?:\.txt)?$", re.IGNORECASE)


def _normalize_index(df):
    out = df.copy()
    out.index = out.index.map(str)
    return out


def read_reference_csv(repo_root, relative_path):
    proc = subprocess.run(
        ["git", "show", f"{REFERENCE_COMMIT}:{relative_path}"],
        cwd=repo_root,
        check=True,
        capture_output=True,
        text=True,
    )
    return _normalize_index(pd.read_csv(StringIO(proc.stdout), index_col=0))


def read_current_csv(repo_root, relative_path):
    return _normalize_index(pd.read_csv(Path(repo_root) / relative_path, index_col=0))


def _passes_length_rule(row):
    if "text_length" not in row.index:
        return True
    L = pd.to_numeric(pd.Series([row.get("text_length")]), errors="coerce").iloc[0]
    return pd.isna(L) or L >= MIN_TEXT_LENGTH


def _arxiv_old_prefix(idx):
    return str(idx).zfill(4)


def _arxiv_current_prefix(idx):
    m = re.match(r"^PAPERS_(\d{4})\.", str(idx), re.IGNORECASE)
    return m.group(1) if m else None


def reconcile_arxiv_cohort(repo_root, relative_path, dataset_name):
    """
    Conservative arXiv reconciliation for 010.

    - The old 005 table defines cohort membership.
    - A YYMM prefix with exactly one current candidate is treated as an unambiguous
      current representation of that old row.
    - A YYMM prefix with multiple current candidates is NOT resolved using NERO
      similarity. The old reference row is retained unchanged.
    - No alternative member of an ambiguous group is called "new".
    - No current-only arXiv row is included in 010 unless provenance later establishes
      that it is genuinely absent from the old cohort.
    """
    ref = read_reference_csv(repo_root, relative_path)
    cur = read_current_csv(repo_root, relative_path)
    out = ref.copy()

    current_groups = {}
    for idx in cur.index:
        p = _arxiv_current_prefix(idx)
        if p is not None:
            current_groups.setdefault(p, []).append(idx)

    matched_map = {}
    ambiguous_groups = {}
    old_without_current = []

    for old_idx in ref.index:
        prefix = _arxiv_old_prefix(old_idx)
        candidates = current_groups.get(prefix, [])
        if len(candidates) == 1:
            matched_map[old_idx] = candidates[0]
        elif len(candidates) > 1:
            ambiguous_groups[old_idx] = list(candidates)
        else:
            old_without_current.append(old_idx)

    n_matched_updated = 0
    matched_below_length_fallback = []
    restored_fields = {}

    if USE_CURRENT_ARXIV_VALUES:
        for old_idx, cur_idx in matched_map.items():
            crow = cur.loc[cur_idx]
            if not _passes_length_rule(crow):
                matched_below_length_fallback.append((old_idx, cur_idx))
                continue

            for col in cur.columns:
                if col not in out.columns:
                    out[col] = np.nan
            for col in cur.columns:
                value = crow.get(col)
                if pd.notna(value):
                    out.at[old_idx, col] = value
                elif col in ref.columns and pd.notna(ref.at[old_idx, col]):
                    restored_fields[col] = restored_fields.get(col, 0) + 1
            out.at[old_idx, "current_arxiv_id"] = cur_idx
            n_matched_updated += 1

    # Deliberately no new arXiv rows in 010.
    current_prefixes = {_arxiv_current_prefix(idx) for idx in cur.index}
    old_prefixes = {_arxiv_old_prefix(idx) for idx in ref.index}
    truly_unmatched_current = [
        idx for idx in cur.index
        if _arxiv_current_prefix(idx) not in old_prefixes
    ]

    audit = {
        "dataset": dataset_name,
        "path": relative_path,
        "reference_rows": len(ref),
        "current_rows": len(cur),
        "matched_reference_rows": len(matched_map),
        "matched_rows_updated": n_matched_updated,
        "matched_below_length_fallback": len(matched_below_length_fallback),
        "ambiguous_prefix_groups": len(ambiguous_groups),
        "new_current_rows_included": 0,
        "old_rows_without_current_match": len(old_without_current),
        "restored_fields": restored_fields,
        "matched_map": matched_map,
        "matched_below_length_pairs": matched_below_length_fallback,
        "ambiguous_groups": ambiguous_groups,
        "old_without_current": old_without_current,
        "current_prefixes_absent_from_old": truly_unmatched_current,
    }
    return out, audit


def reconcile_reference_cohort(
    repo_root,
    relative_path,
    dataset_name,
    *,
    use_current_values=None,
):
    """
    Default non-arXiv replication logic.

    - OLD snapshot defines cohort membership.
    - Matched rows may use current non-missing values.
    - Current-only rows are excluded.
    - Old-only rows are restored.
    - Missing current fields fall back to old values.
    - A current row below MIN_TEXT_LENGTH cannot replace the old row.
    """
    ref = read_reference_csv(repo_root, relative_path)
    cur = read_current_csv(repo_root, relative_path)

    # Only apply the threshold here if the REFERENCE table itself carried it.
    # Otherwise preserve the historical 005 cohort exactly.
    if "text_length" in ref.columns:
        Lref = pd.to_numeric(ref["text_length"], errors="coerce")
        ref = ref.loc[Lref.isna() | (Lref >= MIN_TEXT_LENGTH)].copy()

    ref_ids = set(ref.index)
    cur_ids = set(cur.index)
    matched_ids = sorted(ref_ids & cur_ids)
    restored_row_ids = sorted(ref_ids - cur_ids)
    current_only_ids = sorted(cur_ids - ref_ids)

    out = ref.copy()
    if use_current_values is None:
        use_current_values = USE_CURRENT_REPLICATE_VALUES

    n_length_rejected = 0
    n_rows_updated = 0
    restored_fields = {}

    if use_current_values:
        for idx in matched_ids:
            crow = cur.loc[idx]
            if not _passes_length_rule(crow):
                n_length_rejected += 1
                continue

            for col in cur.columns:
                if col not in out.columns:
                    out[col] = np.nan
            for col in cur.columns:
                value = crow.get(col)
                if pd.notna(value):
                    out.at[idx, col] = value
                elif col in ref.columns and pd.notna(ref.at[idx, col]):
                    restored_fields[col] = restored_fields.get(col, 0) + 1
            n_rows_updated += 1

    rewrite_ids = [idx for idx in out.index if REWRITE_RE.search(str(idx))]

    audit = {
        "dataset": dataset_name,
        "path": relative_path,
        "reference_rows": len(ref),
        "current_rows": len(cur),
        "matched_rows": len(matched_ids),
        "current_only_total": len(current_only_ids),
        "current_only_included": 0,
        "current_only_excluded": len(current_only_ids),
        "old_rows_restored": len(restored_row_ids),
        "current_rows_updated": n_rows_updated,
        "current_below_length_fallback": n_length_rejected,
        "rewrite_rows_in_final_cohort": len(rewrite_ids),
        "restored_fields": restored_fields,
        "restored_row_ids": restored_row_ids,
        "current_only_ids": current_only_ids,
        "rewrite_ids": rewrite_ids,
    }
    return out, audit

## Recognition without training using averged estimate of entropy rate

In [ ]:
from pathlib import Path

_CWD = Path.cwd().resolve()
if (_CWD / "api_data_collection").exists():
    REPO_PATH = _CWD
elif (_CWD.parent / "api_data_collection").exists():
    REPO_PATH = _CWD.parent
else:
    raise RuntimeError("Run this notebook from the nero repository root or notebooks/ directory.")

DATA_PATHS = {
    "human1": ("paper_data/legacy/gutenberg/gutenberg_table.csv", 0, "Gutenberg project (legacy 002)"),
    "human2": ("paper_data/legacy/arxiv_entropy/arxiv_table.csv", 0, "Arxiv papers (legacy 002)"),
    "human3": ("api_data_collection/ai_data/loc-pd/loc-pd_detection.csv", 0, "Library of Congress public domain (003)"),
    "human4": ("api_data_collection/ai_data/us-pd/us-pd_detection.csv", 0, "US public domain (003)"),
    "ai0": ("api_data_collection/ai_data/openai/gpt4o/gpt_4o_detection.csv", 1, "API GPT-4o (003)"),
    "ai1": ("paper_data/legacy/ai_longform_gpt4o/ai_web_longform_table.csv", 1, "AI Generated GPT-4o longform (legacy 002)"),
    "ai2": ("paper_data/legacy/ai_longform_gpt3.5/gpt_35_table.csv", 1, "AI Generated GPT-3.5 longform (legacy 002)"),
    "ai3": ("paper_data/legacy/ai_longform_gpt4.0/gpt_40_table.csv", 1, "AI Generated GPT-4.0 longform (legacy 002)"),
    "ai4": ("api_data_collection/ai_data/claude_sonnet_4/claude_detection.csv", 1, "Claude Sonnet 4 (003)"),
    "ai5": ("api_data_collection/ai_data/gemini_2.5_pro/gemini_25_detection.csv", 1, "Gemini 2.5 Pro (003)"),
    "ai6": ("api_data_collection/ai_data/openai/gpt5/gpt_5_detection.csv", 1, "GPT-5 (003)"),
    "ai7": ("api_data_collection/ai_data/openai/gpt4.0/gpt_4zero_detection.csv", 1, "API GPT-4.0 (003)"),
}

raw_replicates = {}
replication_audit = []
NAMES = {}

for key, (relpath, label, display_name) in DATA_PATHS.items():
    if key == "human2":
        # ARXIV EXCEPTION:
        # Match old YYMM reference rows to current filenames, refresh matched
        # rows when eligible, and add genuinely new current papers >=150k.
        reconciled, audit = reconcile_arxiv_cohort(
            REPO_PATH,
            relpath,
            display_name,
        )
    else:
        reconciled, audit = reconcile_reference_cohort(
            REPO_PATH,
            relpath,
            display_name,
            use_current_values=USE_CURRENT_REPLICATE_VALUES,
        )

    raw_replicates[key] = reconciled
    replication_audit.append(audit)
    NAMES[key] = display_name
    globals()[key] = getDataFrame(reconciled, varname=label)

dallhuman = pd.concat([human1,human2,human3,human4])
df = pd.concat([human1,human2,human3,human4,ai0,ai1,ai2,ai3,ai4,ai5,ai6,ai7])

print("repo:", REPO_PATH)
print("rows entering the replicated 005 analysis:")
print({name: len(globals()[name]) for name in NAMES})


## REPLICATION / IDENTITY AUDIT

The old 005 snapshot defines the replication cohort. Current values may refresh the same identified row, but current-only rows are excluded for non-arXiv datasets.

For arXiv, notebook 010 does **not** use NERO-vector similarity to decide identity. YYMM groups with more than one current paper are left unresolved and retain the old reference row. This avoids circularly using the detector output itself to decide which document is "the same" document.

The 150k threshold is not retroactively used to delete frozen 005 rows that lack old length metadata. It governs current replacements/new candidates. Therefore "005 replication" and "a newly redefined >=150k cohort" are distinct analyses.

In [ ]:
audit_df = pd.DataFrame([
    {k: v for k, v in a.items() if not isinstance(v, (dict, list, tuple))}
    for a in replication_audit
])
display(audit_df)

for a in replication_audit:
    print("\n---", a["dataset"], "---")
    if "matched_map" in a:
        print("ARXIV unambiguous old->current matches:", a["matched_map"])
        print("ambiguous YYMM groups retained as OLD reference rows:", a["ambiguous_prefix_groups"])
        for old_id, candidates in a["ambiguous_groups"].items():
            print("  unresolved:", old_id, "->", candidates)
        print("current prefixes absent from old:", a["current_prefixes_absent_from_old"])
        print("matched rows falling back to old because current <150k:",
              a["matched_below_length_pairs"])
        if a["restored_fields"]:
            print("fields restored from old snapshot:", a["restored_fields"])
    else:
        if a["old_rows_restored"]:
            print("old rows restored:", a["restored_row_ids"])
        if a["current_only_excluded"]:
            print("current-only rows excluded:", a["current_only_ids"])
        if a["rewrite_rows_in_final_cohort"]:
            print("rewrite/perturbation rows:", a["rewrite_ids"])
        if a["restored_fields"]:
            print("fields restored from old snapshot:", a["restored_fields"])

In [ ]:
# Build corrected NERO-derived features from explicit rate columns only.
df_ = df.copy()
rate_cols = [f"rates_{i}" for i in range(EXPECTED_RATE_COMPONENTS)]
missing_rate_cols = [c for c in rate_cols if c not in df_.columns]
if missing_rate_cols:
    raise RuntimeError(f"Missing NERO rate columns: {missing_rate_cols}")

df_["nz_entropy_rate"] = df_[rate_cols].replace(0, np.nan).median(axis=1)
df_["nzmean"] = (df_[rate_cols] != 0).mean(axis=1)
df_["nzstd"] = (df_[rate_cols] != 0).std(axis=1)
df_["nzero"] = (df_[rate_cols] == 0).sum(axis=1)

assert "label" not in rate_cols
assert len(rate_cols) == EXPECTED_RATE_COMPONENTS

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

def detector_auc(frame, score_col, *, positive_label, name):
    """Detector-specific complete-case ROC/AUC. Missing scores are never replaced by zero."""
    d = frame[["label", score_col]].dropna().copy()
    if d["label"].nunique() < 2:
        raise ValueError(f"{name}: both classes are required")
    y = (d["label"].values == positive_label).astype(int)
    score = d[score_col].astype(float).values
    fpr, tpr, thresholds = roc_curve(y, score, pos_label=1)
    return {
        "name": name,
        "n": len(d),
        "positive_label": positive_label,
        "auc": roc_auc_score(y, score),
        "roc": pd.DataFrame({"fpr": fpr, "tpr": tpr, "threshold": thresholds}),
    }

nero_zero = detector_auc(df_, "nz_entropy_rate", positive_label=0, name="NERO zero-shot")
nero_entropy = detector_auc(df_, "entropy_rate", positive_label=0, name="NERO entropy_rate")
howkgpt = detector_auc(df_, "perplexity", positive_label=1, name="HowkGPT perplexity")
zerogpt = detector_auc(df_, "percentage", positive_label=1, name="ZeroGPT percentage")

for r in [nero_zero, nero_entropy, howkgpt, zerogpt]:
    print(f"{r['name']}: AUC={r['auc']:.6f}, n={r['n']}")

# Keep a single real ROC per detector; do not concatenate different score ROCs.
D1 = nero_zero["roc"]
D2 = nero_entropy["roc"]
DP = howkgpt["roc"]
DPc = zerogpt["roc"]

In [ ]:
pd.DataFrame({"label": df_["label"].values}).to_csv("ytrue.csv", index=False)

In [ ]:
# Optional zedstat smoothing/interval for the SINGLE zero-shot NERO ROC.
# Here positive class is human (label 0), so positive_samples must count humans.
from zedstat import zedstat

nero_roc_for_zed = D1[["fpr", "tpr"]].replace([np.inf, -np.inf], np.nan).dropna()
n_human = int((df_.loc[df_["nz_entropy_rate"].notna(), "label"] == 0).sum())

zte = zedstat.processRoc(
    df=nero_roc_for_zed,
    order=3,
    total_samples=nero_zero["n"],
    positive_samples=n_human,
    alpha=0.01,
    prevalence=.5,
)
zte.smooth(STEP=0.001)
zte.allmeasures(interpolate=True)
zte.usample(precision=2)
zte.getBounds()

auc_e = np.array(zte.auc(alpha=.05)) * 100
deltae = (auc_e[1] - auc_e[2]) / 2
le = str(auc_e[0])[:4] + "%"
print("raw sklearn NERO zero-shot AUC:", nero_zero["auc"])
print("zedstat NERO zero-shot AUC:", zte.auc(alpha=.05))

In [ ]:
plt.figure(figsize=(4,4))
ax = plt.gca()
ax.plot(D1["fpr"], D1["tpr"], "--", lw=2)
ax.plot(DP["fpr"], DP["tpr"], "--", lw=2)
ax.plot(DPc["fpr"], DPc["tpr"], "--", lw=2)
ax.set_xlim(0,1)
ax.set_ylim(0,1)
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.legend([
    f"NERO zero-shot AUC: {nero_zero['auc']:.3f}",
    f"Perplexity AUC: {howkgpt['auc']:.3f}",
    f"Percentage AUC: {zerogpt['auc']:.3f}",
], loc="lower right")
ax.set_title(f"Detector ROC curves; detector-specific complete cases")
plt.savefig("roc-notraining1.pdf", bbox_inches="tight", transparent=True)

In [ ]:
#dx=pd.DataFrame({'tpr_upper':zte.df_lim['U'].tpr,'tpr_lower':zte.df_lim['L'].tpr}).join(zte.get()['tpr'])
#dx.to_csv('nero_all_notraining.csv')

In [ ]:
df_all = pd.concat([human1,human2,human3,human4,ai0,ai1,ai2,ai3,ai4,ai5,ai6,ai7])
SUBSET = "ALL"

In [ ]:
# Same corrected feature construction for the ALL cohort.
df_ = df_all.copy()
rate_cols = [f"rates_{i}" for i in range(EXPECTED_RATE_COMPONENTS)]
df_["nz_entropy_rate"] = df_[rate_cols].replace(0, np.nan).median(axis=1)
df_["nzmean"] = (df_[rate_cols] != 0).mean(axis=1)
df_["nzstd"] = (df_[rate_cols] != 0).std(axis=1)
df_["nzero"] = (df_[rate_cols] == 0).sum(axis=1)

In [ ]:
nero_all = detector_auc(df_, "nz_entropy_rate", positive_label=0, name="NERO zero-shot")
perplexity_all = detector_auc(df_, "perplexity", positive_label=1, name="HowkGPT perplexity")
percentage_all = detector_auc(df_, "percentage", positive_label=1, name="ZeroGPT percentage")

print("ALL:")
for r in [nero_all, perplexity_all, percentage_all]:
    print(f"{r['name']}: AUC={r['auc']:.6f}, n={r['n']}")

In [ ]:
# Export detector-specific ROC curves without combining them.
nero_all["roc"].to_csv("nero_ALL_notraining.csv", index=False)
percentage_all["roc"].to_csv("percentage_ALL.csv", index=False)
perplexity_all["roc"].to_csv("perplexity_ALL.csv", index=False)

In [ ]:
plt.figure(figsize=(4,4))
ax = plt.gca()
for r, ls in [(nero_all, "--"), (percentage_all, "--"), (perplexity_all, "--")]:
    ax.plot(r["roc"]["fpr"], r["roc"]["tpr"], ls, lw=2)
ax.set_xlim(0,1)
ax.set_ylim(0,1)
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.legend([
    f"NERO zero-shot: {nero_all['auc']:.3f}",
    f"Percentage: {percentage_all['auc']:.3f}",
    f"Perplexity: {perplexity_all['auc']:.3f}",
], loc="lower right")
plt.savefig("roc-notrainingALL.pdf", bbox_inches="tight", transparent=True)

# AUC distribution for subsets

In [ ]:
import itertools
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

def _auc_complete_cases(df_, score_col, positive_label):
    d = df_[["label", score_col]].dropna()
    if d["label"].nunique() < 2:
        return np.nan
    y = (d["label"].values == positive_label).astype(int)
    return float(roc_auc_score(y, d[score_col].astype(float).values))

def compute_auc_triple_for_df(df_in):
    """
    Three independent detector AUCs on detector-specific complete cases.
    NERO is represented by one scalar zero-shot score: median nonzero rate.
    """
    d = df_in.copy()
    rate_cols = [f"rates_{i}" for i in range(EXPECTED_RATE_COMPONENTS)]
    d["nz_entropy_rate"] = d[rate_cols].replace(0, np.nan).median(axis=1)

    auc_nero = _auc_complete_cases(d, "nz_entropy_rate", positive_label=0)
    auc_percentage = _auc_complete_cases(d, "percentage", positive_label=1)
    auc_perplexity = _auc_complete_cases(d, "perplexity", positive_label=1)
    return auc_nero, auc_percentage, auc_perplexity

def auc_triples_over_subsets(
    dfs,
    *,
    human_keys=("human1","human2","human3","human4"),
    ai_keys=("ai0","ai1","ai2","ai3","ai4","ai5","ai6","ai7"),
    human_subset_sizes=(1,),
    ai_subset_sizes=(1,),
):
    rows = []
    for hs in human_subset_sizes:
        for humans_sel in itertools.combinations(human_keys, hs):
            for asz in ai_subset_sizes:
                for ais_sel in itertools.combinations(ai_keys, asz):
                    chosen = list(humans_sel) + list(ais_sel)
                    d = pd.concat([dfs[k] for k in chosen], axis=0, ignore_index=True)
                    a1, a2, a3 = compute_auc_triple_for_df(d)
                    rows.append({
                        "humans": tuple(humans_sel),
                        "ais": tuple(ais_sel),
                        "n_humans": len(humans_sel),
                        "n_ais": len(ais_sel),
                        "auc_nero": a1,
                        "auc_percentage": a2,
                        "auc_perplexity": a3,
                    })
    return pd.DataFrame(rows)

In [ ]:
%%time
dfs = {
    "human1": human1, "human2": human2, "human3": human3, "human4": human4,
    "ai0": ai0, "ai1": ai1, "ai2": ai2, "ai3": ai3,
    "ai4": ai4, "ai5": ai5, "ai6": ai6, "ai7": ai7,
}

res = auc_triples_over_subsets(
    dfs,
    human_subset_sizes=tuple(range(1, 5)),
    ai_subset_sizes=tuple(range(1, 9))
)

In [ ]:
# These are descriptive sensitivity summaries over heavily overlapping dataset combinations.
# They are NOT sampling confidence intervals.
def descriptive_summary(x):
    x = pd.Series(x, dtype=float).dropna().values
    return {
        "n_subsets": len(x),
        "mean": float(np.mean(x)),
        "median": float(np.median(x)),
        "q01": float(np.quantile(x, 0.01)),
        "q05": float(np.quantile(x, 0.05)),
        "q95": float(np.quantile(x, 0.95)),
        "q99": float(np.quantile(x, 0.99)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }

subset_summary = pd.DataFrame({
    "NERO": descriptive_summary(res.auc_nero),
    "Perplexity": descriptive_summary(res.auc_perplexity),
    "Percentage": descriptive_summary(res.auc_percentage),
}).T
display(subset_summary)

In [ ]:
labels = ["NERO", "Perplexity", "Percentage"]
data = [
    np.asarray(res.auc_nero, dtype=float),
    np.asarray(res.auc_perplexity, dtype=float),
    np.asarray(res.auc_percentage, dtype=float),
]

plt.figure(figsize=(7,5))
rng = np.random.default_rng(0)
for i, x in enumerate(data):
    x = x[~np.isnan(x)]
    jitter = rng.normal(0, 0.05, size=len(x))
    plt.scatter(np.full(len(x), i) + jitter, x, alpha=0.25, s=14, linewidths=0)

plt.xticks(range(3), labels)
plt.ylabel("AUC")
plt.title(f"Descriptive AUC distribution across {len(res)} overlapping human×AI subsets")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
res

In [ ]:
labels = ["NERO", "Perplexity", "Percentage"]
data = [
    np.asarray(res.auc_nero, dtype=float),
    np.asarray(res.auc_perplexity, dtype=float),
    np.asarray(res.auc_percentage, dtype=float),
]
rng = np.random.default_rng(0)
rows = []
for i, (lab, x) in enumerate(zip(labels, data)):
    x = x[~np.isnan(x)]
    jitter = rng.normal(0, 0.05, size=len(x))
    for y, j in zip(x, jitter):
        rows.append({"metric": lab, "x_center": i, "x_jitter": float(j), "x": float(i+j), "auc": float(y)})
df_scatter = pd.DataFrame(rows)
df_scatter.to_csv("auc_scatter_pgf.csv", index=False)

summ_rows = []
for i, lab in enumerate(labels):
    s = descriptive_summary(df_scatter.loc[df_scatter.metric == lab, "auc"].values)
    summ_rows.append({"metric": lab, "x": i, **s})
df_summary = pd.DataFrame(summ_rows)
df_summary.to_csv("auc_summary_pgf.csv", index=False)
display(df_summary)

In [ ]:
df_plot = pd.DataFrame({
    "AUC": np.concatenate([
        np.asarray(res.auc_nero, dtype=float),
        np.asarray(res.auc_perplexity, dtype=float),
        np.asarray(res.auc_percentage, dtype=float),
    ]),
    "Metric": (
        ["NERO"] * len(res.auc_nero)
        + ["Perplexity"] * len(res.auc_perplexity)
        + ["Percentage"] * len(res.auc_percentage)
    )
}).dropna()

plt.figure(figsize=(7,5))
for i, metric in enumerate(["NERO","Perplexity","Percentage"]):
    vals = df_plot.loc[df_plot.Metric == metric, "AUC"].values
    q05, med, q95 = np.quantile(vals, [0.05, 0.5, 0.95])
    plt.scatter(np.full(len(vals), i), vals, alpha=0.08, s=8)
    plt.errorbar(i, med, yerr=[[med-q05],[q95-med]], fmt="o", capsize=6, linewidth=2)
plt.xticks(range(3), ["NERO","Perplexity","Percentage"])
plt.ylabel("AUC")
plt.title("Subset sensitivity: median and 5–95% descriptive range")
plt.tight_layout()
plt.show()

# Classifier 

In [ ]:
def add_nero_features(frame):
    out = frame.copy()
    rate_cols = [f"rates_{i}" for i in range(EXPECTED_RATE_COMPONENTS)]
    out["nz_entropy_rate"] = out[rate_cols].replace(0, np.nan).median(axis=1)
    out["nzmean"] = (out[rate_cols] != 0).mean(axis=1)
    out["nzstd"] = (out[rate_cols] != 0).std(axis=1)
    out["nzero"] = (out[rate_cols] == 0).sum(axis=1)
    return out

CLASSIFIER_FEATURES = (
    ["entropy_rate"]
    + [f"rates_{i}" for i in range(EXPECTED_RATE_COMPONENTS)]
    + ["nz_entropy_rate", "nzmean", "nzstd", "nzero"]
)
assert len(CLASSIFIER_FEATURES) == 46

In [ ]:
# Future-model experiment:
# GPT-5 (ai6) is NEVER used for training.
base_classifier = pd.concat([human1,human2,human3,human4,ai0,ai1,ai2,ai3,ai4,ai5,ai7], ignore_index=True)
future_gpt5 = ai6.copy().reset_index(drop=True)

base_classifier = add_nero_features(base_classifier)
future_gpt5 = add_nero_features(future_gpt5)

# Stratified 50/50 split of the pre-GPT5 pool.
df_train, df_test_in_domain = train_test_split(
    base_classifier,
    test_size=0.5,
    random_state=42,
    stratify=base_classifier["label"],
)

# Final test = held-out pre-GPT5 examples + ALL GPT-5 examples.
df_test = pd.concat([df_test_in_domain, future_gpt5], ignore_index=True)

X_train = df_train[CLASSIFIER_FEATURES].values
y_train = df_train["label"].astype(int).values
X_test = df_test[CLASSIFIER_FEATURES].values
y_test = df_test["label"].astype(int).values

print("train rows:", len(df_train), "GPT-5 rows in train: 0")
print("test rows:", len(df_test), "GPT-5 rows appended:", len(future_gpt5))

In [ ]:
print("Classifier features:", len(CLASSIFIER_FEATURES))
print(CLASSIFIER_FEATURES)

In [ ]:
%%time
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve

# No NERO feature imputation beyond a deterministic zero replacement for missing scalar
# values produced after complete-vector validation. Fit preprocessing on TRAIN only.
imputer = SimpleImputer(strategy="constant", fill_value=0)
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

modelnames = ["randomforest", "gaussianp"]
models = [
    RandomForestClassifier(
        n_estimators=2500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ).fit(X_train_imp, y_train),
    GaussianProcessClassifier(kernel=1.0 * RBF(1.0), random_state=42).fit(X_train_imp, y_train),
]

y_preds = [model.predict_proba(X_test_imp)[:,1] for model in models]
auc_scores = {}
ROC = {}
for modelname, score in zip(modelnames, y_preds):
    fpr, tpr, thresholds = roc_curve(y_test, score, pos_label=1)
    auc_scores[modelname] = roc_auc_score(y_test, score)
    ROC[modelname] = {"fpr": fpr, "tpr": tpr, "thresholds": thresholds}
print(auc_scores)

In [ ]:
plt.figure(figsize=[20,10])
pd.Series(models[0].feature_importances_, index=CLASSIFIER_FEATURES).plot(kind="bar")
plt.ylabel("Random-forest feature importance")
plt.tight_layout()

In [ ]:
# zedstat summary for the trained Gaussian-process classifier ROC.
modeldf_ = [pd.DataFrame(ROC[m]) for m in modelnames]
ztc = zedstat.processRoc(
    df=modeldf_[-1][["fpr","tpr"]],
    order=3,
    total_samples=len(y_test),
    positive_samples=int(np.sum(y_test == 1)),
    alpha=0.01,
    prevalence=.5,
)
ztc.smooth(STEP=0.001)
ztc.allmeasures(interpolate=True)
ztc.usample(precision=2)
ztc.getBounds()
auc_c = np.array(ztc.auc(alpha=.05))*100
lc = str(auc_c[0])[:4] + "%"
print("trained Gaussian-process sklearn AUC:", auc_scores["gaussianp"])
print("trained Gaussian-process zedstat AUC:", ztc.auc(alpha=.05))

In [ ]:
plt.figure(figsize=(4,4))
ax = plt.gca()
ax.plot(nero_all["roc"]["fpr"], nero_all["roc"]["tpr"], "--", lw=2)
ax.plot(percentage_all["roc"]["fpr"], percentage_all["roc"]["tpr"], "--", lw=2)
ax.plot(perplexity_all["roc"]["fpr"], perplexity_all["roc"]["tpr"], "--", lw=2)
ax.plot(ROC["gaussianp"]["fpr"], ROC["gaussianp"]["tpr"], "-", lw=2)

pd.DataFrame(ROC["gaussianp"]).to_csv("nero_ALL_trained.csv", index=False)

ax.set_xlim(0,1)
ax.set_ylim(0,1)
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.legend([
    f"NERO zero-shot: {nero_all['auc']:.3f}",
    f"Percentage: {percentage_all['auc']:.3f}",
    f"Perplexity: {perplexity_all['auc']:.3f}",
    f"NERO trained future-holdout: {auc_scores['gaussianp']:.3f}",
], loc="lower right")
ax.set_title(f"Test samples: {len(y_test)}, AI generated: {int(y_test.sum())}")
plt.savefig("roc-notrainingALL.pdf", bbox_inches="tight", transparent=True)